<br>

#### 리듀서(Reducer)
- **리듀서는 함수형 프로그래밍 개념으로, "기존 값과 새 값을 어떻게 합칠 것인가"를 정의하는 함수**
  - JavaScript의 Redux나 React의 useReducer와 유사한 개념

<br>

1. **기본 리듀서 (덮어쓰기)**
- 새 값이 들어오면 기존 값은 완전히 사라짐
- 가장 단순하고 직관적인 방식

```python
current_step: str
user_id: str
```

<br>

```python
# 초기 상태
state = {"current_step": "시작", "user_id": "user123"}

# 노드 1 실행
update1 = {"current_step": "처리중"}
# 결과: {"current_step": "처리중", "user_id": "user123"}

# 노드 2 실행  
update2 = {"current_step": "완료"}
# 결과: {"current_step": "완료", "user_id": "user123"}
```

<br>

2. **커스텀 리듀서 (`add`를 사용한 누적)**

```python
messages: Annotated[List[str], add]
processing_log: Annotated[List[str], add]
```

- **`Annotated` 타입** : Python 3.9에서 도입된 타입 힌팅 기능
  - 첫 번째 인자: 실제 타입 (List[str])
  - 두 번째 인자: 메타데이터 (리듀서 함수 등)

In [1]:
from operator import add

In [2]:
# 리스트의 경우
result = add([1, 2], [3, 4])  # [1, 2, 3, 4]

# 문자열의 경우
result = add("Hello", " World")  # "Hello World"

# 숫자의 경우
result = add(5, 3)  # 8

<br>

#### 적용 예시
- 초기 상태

In [3]:
state = {
    "messages": ["안녕하세요"],
    "processing_log": ["시작"]
}

- 노드 1 실행
  - 업데이트 후 상태
    - `messages`: ["안녕하세요", "어떻게 도와드릴까요?"]
    - `processing_log`: ["시작", "사용자 인사 감지"]

In [4]:
def node1(state):
    return {
        "messages": ["어떻게 도와드릴까요?"],
        "processing_log": ["사용자 인사 감지"]
    }

- 노드 2 실행
  - 최종 상태
    - `messages`: ["안녕하세요", "어떻게 도와드릴까요?", "무엇을 도와드릴까요?"]
    - `processing_log`: ["시작", "사용자 인사 감지", "응답 생성 완료"]

In [5]:
def node2(state):
    return {
        "messages": ["무엇을 도와드릴까요?"],
        "processing_log": ["응답 생성 완료"]
    }

<br>

### 상태 스키마 설계
- 상태 스키마는 LangGraph에서 데이터 흐름의 청사진
- 절한 스키마 설계는 시스템의 안정성과 유지보수성을 크게 향상

<br>

#### `TypedDict` (권장)
- 가장 일반적이고 권장되는 방법
- **장점**:
  - 가볍고 빠른 성능
  - Python 표준 라이브러리만 사용
  - LangGraph의 기본 스키마 방식
  - 런타임 오버헤드 없음
- **단점**:
  - 런타임 유효성 검사 없음
  - 기본값 설정이 불편함
  - **`NotRequired`로 선택적 필드 정의**

In [6]:
from typing_extensions import TypedDict, NotRequired
from typing import Annotated
from operator import add

In [7]:
class MyState(TypedDict):
    query: str                          # 필수 필드
    results: Annotated[list[str], add]  # 필수 (리듀서 적용)
    count: NotRequired[int]             # 선택적 필드 (노드에서 생략 가능)


<br>

#### `dataclass`
- **장점**:
  - 기본값 설정이 쉬움
  - 자동 `init`, `repr` 생성
  - TypedDict와 유사한 성능

- **단점**:
  - 런타임 유효성 검사 없음

In [8]:
from dataclasses import dataclass, field
from typing import Annotated
from operator import add

In [9]:
@dataclass
class MyState:
    query: str = ""
    results: Annotated[list[str], add] = field(default_factory=list)
    count: int = 0

<br>

#### `Pydantic BaseModel`
- 엄격한 데이터 검증이 필요할 때 사용
- **장점**
  - 런타임 데이터 검증
  - 풍부한 유효성 검사 옵션
  - 자동 타입 변환
- **단점**
  - `TypedDict`보다 느림 (검증 오버헤드)
  - 추가 의존성 필요


In [10]:
from pydantic import BaseModel, Field
from typing import Annotated
from operator import add

In [11]:
class MyState(BaseModel):
    query: str = ""
    results: Annotated[list[str], add] = Field(default_factory=list)
    count: int = Field(default=0, ge=0)  # 0 이상만 허용

<br>

#### 상태 스키마 설계 선택 가이드

<table>
<thead>
<tr>
<th>상황</th>
<th>추천 방법</th>
</tr>
</thead>
<tbody>
<tr>
<td>일반적인 경우</td>
<td>TypedDict</td>
</tr>
<tr>
<td>기본값 필요</td>
<td>dataclass</td>
</tr>
<tr>
<td>엄격한 검증 필요</td>
<td>Pydantic</td>
</tr>
<tr>
<td>고성능 필요</td>
<td>TypedDict</td>
</tr>
<tr>
<td>필드별 선택 여부 구분</td>
<td>TypedDict + NotRequired</td>
</tr>
</tbody>
</table>

<br>

### 스키마 설계 원칙

<br>

#### 명확성 원칙
- **각 필드의 목적과 사용법이 명확하게**

  - 의미 있는 필드명 사용: `data` -> `user_profile`
  - 타입 힌팅 활용: `messages: list -> messages: List[str]`
  - 문서화 추가: `docstring`이나 주석으로 각 필드 설명

<br>

#### 캡슐화 원칙
- **내부 처리용 데이터와 외부 인터페이스를 구분**
- LangGraph에서는 입력/출력 스키마를 분리하여 내부 구현을 숨기고 명확한 인터페이스를 제공

<br>

#### 단일 책임 원칙
- **각 상태 스키마는 하나의 명확한 목적을 가져야 함. 하나의 스키마가 너무 많은 책임을 가지면 복잡도가 증가하고 변경이 어려움**

<br>


### 상태 스키마의 유형

<br>

#### 기본 스키마 (Single Schema)
- **입력과 출력이 동일한 단일 스키마를 사용하는 가장 기본적인 형태**
  - 간단한 챗봇이나 Q&A 시스템
  - 상태가 복잡하지 않은 선형적 워크플로우
  - 프로토타이핑이나 학습 목적

In [12]:
from typing_extensions import TypedDict
from typing import Annotated
from operator import add

In [13]:
class BasicState(TypedDict):
    user_input: str
    ai_response: str
    conversation_history: Annotated[list[str], add]

In [14]:
def chatbot_node(state: BasicState) -> dict:
    response = f"'{state['user_input']}'에 대한 응답입니다."
    return {
        "ai_response": response,
        "conversation_history": [f"User: {state['user_input']}", f"AI: {response}"]
    }

<br>

#### 명시적 입출력 스키마 (Explicit Input/Output Schema)
- **입력과 출력을 별도로 정의하여 인터페이스를 명확하게 제어**
  - API 같은 명확한 인터페이스가 필요한 경우
  - 외부 시스템과 연동할 때
  - 내부 처리 과정을 숨기고 싶을 때

<br>


In [15]:
class InputState(TypedDict):
    question: str
    
class OutputState(TypedDict):
    answer: str

class OverallState(InputState, OutputState):
    intermediate_data: str
    search_results: list[str]

In [16]:
def search_node(state: InputState) -> dict:
    return {
        "search_results": ["결과1", "결과2"],
        "intermediate_data": f"'{state['question']}' 검색 완료"
    }

def answer_node(state: OverallState) -> OutputState:
    return {"answer": f"검색 결과: {state['search_results'][0]}"}


- StateGraph 설정

In [18]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [19]:
builder = StateGraph(
   OverallState,
   input_schema=InputState,
   output_schema=OutputState
)

<br>

#### 다중 스키마 + PrivateState
- **내부 노드 간 통신을 위한 private 스키마를 포함하는 복잡한 구조**
  - RAG 파이프라인
  - 다단계 처리가 필요한 복잡한 시스템
  - 각 단계별로 다른 데이터 구조가 필요한 경우
- **노드는 그래프 초기화 시 정의되지 않은 새로운 상태 채널(PrivateState의 필드)을 추가할 수 있음 $\rightarrow$ 이를 통해 내부 처리 데이터를 외부에 노출하지 않으면서 노드 간에 전달할 수 있음**

In [20]:
class OverallState(TypedDict):
    question: str
    answer: str

class PrivateState(TypedDict):
    internal_score: float
    debug_info: str

In [21]:
def internal_node(state: OverallState) -> PrivateState:
    # 노드는 OverallState에 정의되지 않은 필드도 반환 가능
    return {
        "internal_score": 0.95,
        "debug_info": "내부 처리 완료"
    }


<br>

### 리듀서(Reducer) - 상태 업데이트
- 리듀서는 LangGraph에서 상태 업데이트 로직을 정의하는 핵심 메커니즘
- **각 노드가 반환하는 업데이트를 기존 상태에 어떻게 적용할지 결정하는 함수**
- **리듀서의 필요성**
  - 여러 노드가 동시에 또는 순차적으로 실행될 때, 각 노드의 출력을 상태에 통합하는 방법이 필요
  - 단순히 덮어쓰기만 하면 이전 정보가 손실될 수 있으며, 리듀서를 통해 누적, 병합, 최대값 유지 등 다양한 업데이트 전략을 구현할 수 있음
- **노드는 전체 상태가 아닌 업데이트할 부분만 반환하며, 리듀서는 이 부분 업데이트를 기존 상태와 결합**

<br>

#### 기본 동작 원리

```python
# 리듀서 없음 = 덮어쓰기 (Default Reducer)
old_state = {"counter": 5}
new_update = {"counter": 10}
result = {"counter": 10}  # 기존 값이 완전히 대체됨

# 리듀서 있음 = 사용자 정의 동작
from operator import add
old_state = {"items": [1, 2, 3]}
new_update = {"items": [4, 5]}
result = {"items": [1, 2, 3, 4, 5]}  # 리스트가 연결됨
```

<br>

### 기본 리듀서

<br>

#### 덮어쓰기 (Default)


In [22]:
from typing_extensions import TypedDict

In [23]:
class SimpleState(TypedDict):
    counter: int          # 기본 리듀서: 덮어쓰기
    current_user: str     # 기본 리듀서: 덮어쓰기
    status: str          # 기본 리듀서: 덮어쓰기

In [24]:
def update_counter(state: SimpleState) -> SimpleState:
    """
    이 노드는 counter 값만 업데이트
    다른 키들(current_user, status)은 변경되지 않음
    """
    return {"counter": state["counter"] + 1}

In [25]:
def update_multiple(state: SimpleState) -> SimpleState:
    """
    여러 키를 동시에 업데이트할 수 있음
    반환하지 않은 키는 그대로 유지.
    """
    return {
        "counter": 0,           # counter를 0으로 리셋
        "status": "completed"   # status를 "completed"로 변경
    }

<br>

### 내장 리듀서

<br>

#### `operator.add` - 리스트/문자열 추가

In [26]:
from operator import add
from typing import Annotated

In [27]:
class AddState(TypedDict):
    # Annotated[타입, 리듀서함수] 형식으로 리듀서 지정
    messages: Annotated[list[str], add]      # 리스트에 추가
    log_text: Annotated[str, add]            # 문자열 연결

In [28]:
def add_message(state: AddState) -> AddState:
    """
    add 리듀서의 동작:
    - 리스트: 기존_리스트 + 새_리스트
    - 문자열: 기존_문자열 + 새_문자열
    """
    return {
        "messages": ["새 메시지"],              
        "log_text": "새 로그 항목\n"            
    }

In [29]:
initial_state = {
    "messages": ["안녕하세요"],
    "log_text": "시작\n"
}

- `add_message` 노드 실행 후

```python
{
    "messages": ["안녕하세요", "새 메시지"],
    "log_text": "시작\n새 로그 항목\n"
}
```

<br>

#### `add_messages` : 메시지 전용 리듀서
- LangChain 메시지 객체를 위한 특별한 리듀서

In [30]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from typing import Annotated

In [31]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # 메시지 전용 리듀서

In [32]:
def process_user_input(state: ChatState) -> ChatState:
    """
    add_messages 리듀서의 특징:
    1. 새 메시지는 리스트에 추가
    2. 동일한 ID를 가진 메시지는 업데이트
    3. 자동으로 메시지 객체로 역직렬화
    """
    return {
        "messages": [HumanMessage(content="사용자 입력", id="msg_001")]
    }

<br>

- **혹은 `MessageState` 사용**

In [33]:
from langgraph.graph import MessagesState

In [34]:
class MyState(MessagesState):
    """MessagesState를 상속받으면 messages 필드와 add_messages 리듀서가 자동 포함"""
    documents: list[str]  # 추가 필드 정의 가능

<br>

### 사용자 정의 리듀서
- 특정 비즈니스 로직을 구현하는 리듀서

In [35]:
def max_reducer(existing: int, new: int) -> int:
    """
    더 큰 값을 유지하는 리듀서

    Args:
        existing: 현재 상태의 값
        new: 노드가 반환한 새 값

    Returns:
        둘 중 더 큰 값
    """
    return max(existing or 0, new or 0)

In [36]:
class MaxState(TypedDict):
    highest_score: Annotated[int, max_reducer]
    current_score: int  # 리듀서 없음 (덮어쓰기)

<br>

- **`update_score` 실행 후:**
  - `{"highest_score": 90, "current_score": 85}`
  
  $\rightarrow$ `highest_score`는 90이 더 크므로 유지, `current_score`는 85로 덮어쓰기

In [37]:
def update_score(state: MaxState) -> MaxState:
    new_score = 85
    return {
        "highest_score": new_score,  # max_reducer가 적용됨
        "current_score": new_score   # 단순 덮어쓰기
    }

<br>

#### 복잡한 사용자 정의 리듀서
- 딕셔너리 병합

In [38]:
def merge_dict_reducer(existing: dict, new: dict) -> dict:
    """
    딕셔너리를 깊게 병합하는 리듀서
    중첩된 딕셔너리도 재귀적으로 병합
    """
    if not existing:
        return new or {}
    if not new:
        return existing

    result = existing.copy()
    for key, value in new.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            # 중첩된 딕셔너리는 재귀적으로 병합
            result[key] = merge_dict_reducer(result[key], value)
        else:
            # 그 외의 경우는 덮어쓰기
            result[key] = value
    return result

In [39]:
class ConfigState(TypedDict):
    settings: Annotated[dict, merge_dict_reducer]

In [40]:
def update_ui_settings(state: ConfigState) -> ConfigState:
    """UI 설정만 업데이트"""
    return {
        "settings": {
            "ui": {"theme": "dark"}  # ui.theme만 변경
        }
    }

In [41]:
def update_performance_settings(state: ConfigState) -> ConfigState:
    """성능 설정만 업데이트"""
    return {
        "settings": {
            "performance": {"cache_size": 1000}  # performance.cache_size만 변경
        }
    }

<br>

- **실행 시나리오**

1. 초기 상태: `{"settings": {"ui": {"theme": "light", "font": "Arial"}, "performance": {"cache_size": 500}}}`
2. `update_ui_settings` 실행 후:
- `{"settings": {"ui": {"theme": "dark", "font": "Arial"}, "performance": {"cache_size": 500}}}` $\rightarrow$ `ui.theme`만 변경되고 나머지는 유지
